# 📊 Choosing the right chart

A chart is an *argument*. The same numbers can be made to look reassuring or alarming depending on how you draw them — so choosing a chart is a decision with consequences, not a matter of taste.

This lesson walks through the six charts you will use 95% of the time, what each is **good at**, what each **hides**, and how charts mislead — on purpose or by accident.

**The one question that decides everything:** *what am I comparing?*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("data/eu_media_habits.csv")
BLUE = "#1d3fbf"
df.head(3)

## 1. Bar chart — comparing categories

*Use when:* you have distinct groups and want to compare an amount across them.

In [ ]:
counts = df["main_news_source"].value_counts()

counts.plot(kind="bar", color=BLUE)
plt.ylabel("Number of respondents")
plt.title("Where people get their news")
plt.xticks(rotation=30, ha="right")
plt.show()

✅ **Good for:** precise comparison — humans judge *length* very accurately.

⚠️ **Watch out:** bars must start at zero (see the last section), and beyond ~10 categories they turn into a picket fence. With many categories, use a horizontal bar chart (`kind="barh"`) so the labels stay readable.

## 2. Histogram — the shape of one number

*Use when:* you have one numeric column and want to know how it is distributed — where the bulk sits, whether it is skewed, whether there are outliers.

In [ ]:
df["age"].plot(kind="hist", bins=20, color=BLUE, edgecolor="white")
plt.xlabel("Age")
plt.title("Age distribution of respondents")
plt.show()

### ⚠️ The bin trap — the same data, three stories

A histogram's shape depends on a choice *you* make: how many bins. Run this and watch the same column tell three different stories.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, b in zip(axes, [4, 20, 90]):
    ax.hist(df["age"], bins=b, color=BLUE, edgecolor="white")
    ax.set_title(f"bins = {b}")
    ax.set_xlabel("Age")
plt.tight_layout()
plt.show()

**Left:** too few bins — real structure is smoothed away into blocks. **Right:** too many — random noise looks like meaningful spikes. **Middle:** about right.

There is no universally correct number. The honest habit: **try several before you believe any of them.**

✅ **Good for:** seeing skew, gaps and outliers in one variable.
⚠️ **Watch out:** the bin count changes the story; a histogram of a *category* is meaningless — use a bar chart.

## 3. Box plot — comparing distributions across groups

*Use when:* you want to compare a numeric variable across several categories at once. Each box shows the median (line), the middle 50% (box), the range (whiskers) and outliers (dots).

In [ ]:
df.boxplot(column="hours_social", by="country", figsize=(10, 4), grid=False)
plt.suptitle("")
plt.title("Daily social media hours by country")
plt.xlabel("")
plt.ylabel("Hours per day")
plt.xticks(rotation=30, ha="right")
plt.show()

Eight distributions compared in one glance — that is what box plots are for.

### ⚠️ What box plots hide

A box plot summarises a distribution into five numbers. If the real shape is unusual, those five numbers can hide it completely. Here is an invented example that proves it:

In [ ]:
rng = np.random.default_rng(0)
# two very separate groups of people, combined into one column
bimodal = np.concatenate([rng.normal(25, 4, 300), rng.normal(65, 4, 300)])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].boxplot(bimodal, orientation="horizontal")
axes[0].set_title("Box plot: looks like one ordinary group")
axes[1].hist(bimodal, bins=40, color=BLUE, edgecolor="white")
axes[1].set_title("Histogram: actually two separate groups!")
plt.tight_layout()
plt.show()

The box plot shows a tidy median around 45 — an age **almost nobody in the data actually has**. The histogram reveals the truth instantly.

✅ **Good for:** comparing many groups compactly; spotting outliers.
⚠️ **Watch out:** hides multi-peaked (bimodal) data. Pair it with a histogram when a group matters.

## 4. Scatter plot — the relationship between two numbers

*Use when:* you want to know whether two numeric variables move together.

In [ ]:
df.plot(kind="scatter", x="hours_social", y="trust_news", color=BLUE)
plt.title("Social media hours vs. trust in news")
plt.show()

### ⚠️ Overplotting — when dots hide dots

`trust_news` is a whole number 1–10, so hundreds of people land on the *exact same spot*. A solid dot might be one person or fifty — you cannot tell. Two fixes: make dots transparent (`alpha`), and add a little random *jitter* to separate them.

In [ ]:
jitter = np.random.default_rng(1).normal(0, 0.12, len(df))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df["hours_social"], df["trust_news"], color=BLUE)
axes[0].set_title("Raw: dots stack invisibly")
axes[1].scatter(df["hours_social"], df["trust_news"] + jitter, color=BLUE, alpha=0.25)
axes[1].set_title("alpha + jitter: density becomes visible")
for ax in axes:
    ax.set_xlabel("Hours social media")
    ax.set_ylabel("Trust in news")
plt.tight_layout()
plt.show()

Now you can actually see where people cluster — and the downward drift.

✅ **Good for:** relationships, clusters, outliers.
⚠️ **Watch out:** overplotting with many or rounded values; and a visible pattern is **correlation, never proof of causation**.

## 5. Line chart — change along an ordered scale

*Use when:* the x-axis has a natural order — time, age, temperature. The line implies *"these points connect"*, so the order must be real.

In [ ]:
# group ages into 5-year bands so the line is not pure noise
by_age = df.groupby(df["age"] // 5 * 5)["hours_social"].mean()

by_age.plot(marker="o", color=BLUE)
plt.xlabel("Age group")
plt.ylabel("Average hours of social media")
plt.title("Social media use declines with age")
plt.show()

✅ **Good for:** trends over time or any ordered scale.
⚠️ **Watch out:** never use a line for unordered categories — connecting *Iceland → Austria → Germany* implies a journey that does not exist. That is a bar chart's job.

## 6. Pie chart — the one to avoid

*Use when:* almost never. Here is the same data as a pie and as a bar — try to rank the slices by eye before you look at the bars.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].pie(counts.values, labels=counts.index, autopct="%1.0f%%")
axes[0].set_title("Pie: can you rank these?")
axes[1].barh(counts.index[::-1], counts.values[::-1], color=BLUE)
axes[1].set_title("Bar: instantly obvious")
plt.tight_layout()
plt.show()

People compare **lengths** accurately and **angles** poorly — that is a fact about human vision, not a style preference. The bar chart answers in a fraction of a second.

✅ **Tolerable for:** two or three slices of an obvious whole.
⚠️ **Watch out:** more than three slices, or slices of similar size — use a bar chart.

## 📐 How charts lie: the truncated axis

This is the most common trick in published media, and you should be able to spot it in a second. Below is **exactly the same data**, drawn twice.

In [ ]:
trust = df.groupby("country")["trust_news"].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
trust.plot(kind="bar", ax=axes[0], color=BLUE)
axes[0].set_ylim(0, 10)
axes[0].set_title("Honest: axis starts at 0")
trust.plot(kind="bar", ax=axes[1], color="#c0392b")
axes[1].set_ylim(4.8, 7.6)
axes[1].set_title("Misleading: axis starts at 4.8")
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("Mean trust in news")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**Left:** modest differences between countries. **Right:** one country looks like it has almost no trust at all. Same numbers. Only the axis changed.

For a **bar chart the axis must start at zero**, because the bar's *length* is the message. (For line charts a trimmed axis is often legitimate — the message there is the *slope*.)

This matters directly to EURIDICE's focus on **Communication & Media**: recognising this trick is a media-literacy skill, and *not* using it is a professional obligation.

## 🧭 Your decision table

| What are you asking? | Chart | Main risk |
|---|---|---|
| How do categories compare? | **Bar** | Truncated axis exaggerates |
| What does one variable look like? | **Histogram** | Bin count changes the story |
| How do groups compare in spread? | **Box plot** | Hides multi-peaked data |
| Do two numbers move together? | **Scatter** | Overplotting; ≠ causation |
| How does something change over time? | **Line** | Only for ordered x-axes |
| What share of a whole? | **Bar** (rarely pie) | Angles are hard to compare |

## Your turn

- Draw a bar chart of `household_income_band` — what do you notice about the total?
- Make a box plot of `hours_tv` by `main_news_source`. Which group watches most?
- Redraw the trust chart with `set_ylim(5.5, 7.0)`. How dishonest can you make it look?

## ✅ What you learned

Six chart types and what each is for · the bin trap · what box plots hide · overplotting and how to fix it · why pie charts fail · **how a truncated axis lies**.

📚 **Go further:**
- [From Data to Viz](https://www.data-to-viz.com/) — pick a chart from your data type
- [Python Graph Gallery](https://python-graph-gallery.com/) — copy-paste recipes
- [Matplotlib tutorials](https://matplotlib.org/stable/tutorials/index.html) — the official guide

**Next:** [Seeing what's missing] — before you chart anything, find out what your data *isn't* telling you.